# Combined 3D Force-Chain Architecture Plotter

This notebook plots both the force-colored and EBC-colored versions.

Pipeline:
1. Force-weighted contact network w=1/F
2. Identification of critical edges using Edge Betweenness Centrality (EBC): the fraction of all shortest paths that pass through an edge
3. Backbone identification using change-point detection (PELT). PELT resulted in a backbone of around 3-4% of chains. Here we use a backbone of 10% for visualization, but we can change this.
4. Force chain structural characterization: Quasi-linear force chains connect neighbors with angles less than 45 degrees deviation


It runs the graph analysis once, then produces:

1. **Plot 1:** selected chains colored by contact force.
2. **Plot 2:** selected chains colored by edge betweenness centrality (EBC).
3. ***Quick replot:*** reuse the same `pos`, `G`, `ebc`, and `chains` objects with different display settings, without recomputing EBC.

Edit only the **Settings** cell first.
Run the whole notebook >>
Use quick replot to test different visualization settings

In [1]:
# 
# Imports
# 

from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import networkx as nx

import plotly.graph_objects as go
import matplotlib.cm as cm
import matplotlib.colors as mcolors

In [2]:
# 
# SETTINGS
# 

#  Input files 

particle_file = "dump_mu0.1_mur0.0_mut0.0_01jun26.cor" 
contact_file = "dump_mu0.1_mur0.0_mut0.0_01jun26.lst"

#particle_file = "dump_mu0.7_mur0.1_mut0.1_01jun26.cor" 
#contact_file = "dump_mu0.7_mur0.1_mut0.1_01jun26.lst"

# Output files
output_dir = Path("force_chain_output")
output_force_html = "force_chains_force_colored.html"
output_ebc_html = "force_chains_ebc_colored.html"
output_png = None   # use a filename for PNG export

#mapping
#compute c_forces all pair/local fx fy fz force p1 p2 p3 p4 p8 p9 cutoff radius
# fx, fy, fz, force : normal force components and magnitude
# p1, p2, p3, p4 : sliding force components and magnitude
# p8 rolling force magnitude
# p9 twisiting torque magnitude

# Contact-file columns
# zero-based
# Dump format:
# index, i, j, fx, fy, fz, p1, p2, p3, force, ...
contact_i_col = 1
contact_j_col = 2
contact_force_col = 6   # force = normal force magnitude

# Particle-file columns
# zero-based
# Dump format:
# id type x y z ...
atom_id_col = 0
atom_x_col = 2
atom_y_col = 3
atom_z_col = 4

#  EBC settings
# 90 keeps the top 10% by EBC.
chain_percentile = 85.0

# Maximum local bend angle (deviation from straight path) allowed when walking backbone edges into chains.
angle_limit_deg = 55.0

# Minimum number of particles required for a path to count as a chain.
min_chain_length = 3

# Visualization
# Fraction of gray background contacts to draw.
# 1.0 = all background contacts, 0.25 = 25%
background_contact_fraction = 1.0
random_seed = 1

gray_line_width = 1.0
gray_opacity = 0.35
chain_line_width = 5.0

# Color transforms for selected chain edges.
# Choose "log10" for base-10 log or "ln" for natural log.
force_log_base = "ln"
ebc_log_base = "ln"

# Crop z range for pile view. Use None to show full z range.
z_range = (0, 10)

# Periodic-boundary
# LAMMPS boundary: p p fm, so x and y are periodic, z is not.
# Wrapped contacts across x/y look like long lines in visualization -- need to be removed
remove_periodic_wrap_edges = True
periodic_axes = ("x", "y")

# Reading box bounds from the particle dump. If unavailable, set manually.
# manual_box_bounds = {"x": (0.0, 60.0), "y": (0.0, 60.0), "z": (0.0, 60.0)}
manual_box_bounds = None

# Wraparound edge if |dx| > fraction*Lx or |dy| > fraction*Ly.
# 0.50 = cutoff for periodic wrapping.
periodic_wrap_fraction = 0.50

In [3]:

PositionDict = Dict[int, np.ndarray] #position dictionary
EdgeKey = Tuple[int, int] #edges from-to


def edge_key(u: int, v: int) -> EdgeKey:
    return tuple(sorted((int(u), int(v))))


def unit_vector(vec: np.ndarray) -> np.ndarray:
    return vec / (np.linalg.norm(vec) + 1e-12)


def bend_angle_deg(pos_a: np.ndarray, pos_b: np.ndarray, pos_c: np.ndarray) -> float:
    """
    Local bend angle between segments a->b and b->c.

    0 degrees means locally straight.
    Larger angles mean sharper bending.
    """
    v1 = unit_vector(pos_b - pos_a)
    v2 = unit_vector(pos_c - pos_b)
    dot = np.clip(np.dot(v1, v2), -1.0, 1.0)
    return float(np.degrees(np.arccos(dot)))







    ########
# Position dictionary:
# particle ID -> [x, y, z]
positions = {}

# Edge key:
# (particle1, particle2)
# smaller id first
def edge_key(u, v):
    if u < v:
        return (u, v)
    else:
        return (v, u)


# Unit vector
def unit_vector(vec):
    length = np.linalg.norm(vec)
    # no division by zero
    if length == 0:
        return vec

    return vec / length


# Angle between two connected segments (in degrees)
# deviation from straight

def bend_angle_deg(pos_a, pos_b, pos_c):

    v1 = unit_vector(pos_b - pos_a)
    v2 = unit_vector(pos_c - pos_b)

    # Dot product
    dot = np.dot(v1, v2)

    # Keep within valid range (because of problems with rounding error)
    dot = np.clip(dot, -1.0, 1.0)

    # angle
    angle = np.degrees(np.arccos(dot))

    return angle

In [4]:
def read_lammps_positions(filename: str) -> PositionDict:
    
    positions: PositionDict = {}
    reading_atoms = False

    with open(filename, "r") as f:
        for raw_line in f:
            line = raw_line.strip()

            if line.startswith("ITEM: ATOMS"):
                reading_atoms = True
                continue

            if line.startswith("ITEM:") and not line.startswith("ITEM: ATOMS"):
                reading_atoms = False
                continue

            if reading_atoms and line:
                parts = line.split()
                needed = max(atom_id_col, atom_x_col, atom_y_col, atom_z_col)

                if len(parts) <= needed:
                    continue

                pid = int(float(parts[atom_id_col]))

                positions[pid] = np.array([
                    float(parts[atom_x_col]),
                    float(parts[atom_y_col]),
                    float(parts[atom_z_col]),
                ], dtype=float)

    if not positions:
        raise ValueError(f"No particle positions found in {filename!r}.")

    return positions

def read_lammps_box_bounds(filename: str):
    #Read box bounds from dump file.

    bounds = []
    read_next = 0

    with open(filename, "r") as f:
        for raw_line in f:
            line = raw_line.strip()

            if line.startswith("ITEM: BOX BOUNDS"):
                read_next = 3
                bounds = []
                continue

            if read_next > 0:
                parts = line.split()
                if len(parts) >= 2:
                    bounds.append((float(parts[0]), float(parts[1])))
                read_next -= 1

                if read_next == 0 and len(bounds) == 3:
                    return {"x": bounds[0], "y": bounds[1], "z": bounds[2]}

    return None

In [5]:
def read_lammps_local_contacts(filename: str) -> pd.DataFrame:
    #Output columns:
    #i = first particle id
    #j = second particle id
    #F = chosen force magnitude
    
    rows = []
    reading_entries = False

    with open(filename, "r") as f:
        for raw_line in f:
            line = raw_line.strip()

            if line.startswith("ITEM: ENTRIES"):
                reading_entries = True
                continue

            if line.startswith("ITEM:") and not line.startswith("ITEM: ENTRIES"):
                reading_entries = False
                continue

            if reading_entries and line:
                rows.append(line.split())

    if not rows:
        raise ValueError(f"No rows in {filename!r}.")

    raw = pd.DataFrame(rows).astype(float)

    needed = max(contact_i_col, contact_j_col, contact_force_col)
    if raw.shape[1] <= needed:
        raise ValueError(
            f"Contact file has {raw.shape[1]} columns, "
            f"but requested column index {needed}."
        )

    contacts = pd.DataFrame({
        "i": raw[contact_i_col].astype(int),
        "j": raw[contact_j_col].astype(int),
        "F": raw[contact_force_col].astype(float),
    })

    # remove inf
    contacts = contacts.replace([np.inf, -np.inf], np.nan).dropna()
    contacts = contacts[contacts["F"] > 0].copy()

    return contacts

In [6]:
#Graph and selected edges

def remove_periodic_boundary_edges(
    contacts: pd.DataFrame,
    positions: PositionDict,
    box_bounds: dict | None,
    periodic_axes=("x", "y"),
    wrap_fraction: float = 0.50, 
    #note:any contact whose separation is more thaprint(box_bounds)n half the box length is assumed to cross the periodic boundary
) -> pd.DataFrame:
    
    if box_bounds is None:
        print("WARNING: No box bounds found. Skipping periodic wrap removal.")
        return contacts.copy()

    axis_index = {"x": 0, "y": 1, "z": 2}
    axis_lengths = {}

    for ax in periodic_axes:
        lo, hi = box_bounds[ax]
        L = float(hi - lo)
        if L <= 0:
            raise ValueError(f"Invalid {ax}-box length: {L}")
        axis_lengths[ax] = L

    keep_rows = []
    removed_rows = []

    for idx, row in contacts.iterrows():
        i = int(row["i"])
        j = int(row["j"])

        if i not in positions or j not in positions:
            # Keep it for now; missing positions will be handled elsewhere if needed.
            keep_rows.append(idx)
            continue

        pi = positions[i]
        pj = positions[j]

        crosses_periodic_boundary = False
        for ax in periodic_axes:
            k = axis_index[ax]
            if abs(float(pi[k] - pj[k])) > wrap_fraction * axis_lengths[ax]:
                crosses_periodic_boundary = True
                break

        if crosses_periodic_boundary:
            removed_rows.append(idx)
        else:
            keep_rows.append(idx)

    filtered = contacts.loc[keep_rows].copy().reset_index(drop=True)

    print("Periodic wrap-edge filtering:")
    print(f"  Periodic axes: {periodic_axes}")
    print(f"  Box bounds: {box_bounds}")
    print(f"  Wrap cutoff fraction: {wrap_fraction}")
    print(f"  Before: {len(contacts):,} contacts")
    print(f"  Removed: {len(removed_rows):,} wraparound contacts")
    print(f"  After:  {len(filtered):,} contacts")

    return filtered


def build_contact_graph(contacts: pd.DataFrame) -> nx.Graph:

        #force  = contact force magnitude
        #weight = 1 / force


    G = nx.Graph()

    for row in contacts.itertuples(index=False):
        u = int(row.i)
        v = int(row.j)
        F = float(row.F)

        if u == v or F <= 0:
            continue

        G.add_edge(u, v, force=F, weight=1.0 / F)

    if G.number_of_edges() == 0:
        raise ValueError("Graph has no valid contact edges.")

    return G


def compute_edge_betweenness(G: nx.Graph) -> Dict[EdgeKey, float]:
    ebc_raw = nx.edge_betweenness_centrality(G, weight="weight")
    return {edge_key(u, v): float(value) for (u, v), value in ebc_raw.items()}


def make_backbone_graph(G: nx.Graph, ebc: Dict[EdgeKey, float], percentile: float):
    #keep edges with high EBC
    values = np.array(list(ebc.values()), dtype=float)
    threshold = float(np.percentile(values, percentile))

    H = nx.Graph()

    for u, v, data in G.edges(data=True):
        k = edge_key(u, v)

        if ebc.get(k, 0.0) >= threshold:
            H.add_edge(u, v, **data, ebc=ebc[k])

    return H, threshold

#check wrap edges
print(box_bounds)
Lx = box_bounds["x"][1] - box_bounds["x"][0]
Ly = box_bounds["y"][1] - box_bounds["y"][0]

for c in chains:
    for a, b in zip(c[:-1], c[1:]):
        dx = abs(pos[a][0] - pos[b][0])
        dy = abs(pos[a][1] - pos[b][1])

        if dx > 0.5 * Lx or dy > 0.5 * Ly:
            print("WRAP CHAIN EDGE:", a, b, "dx=", dx, "dy=", dy)

In [7]:
#Edge walk

def choose_next_node(H, positions, chain, direction, angle_limit_deg):
    tip = chain[-1] if direction == "head" else chain[0]
    prev = chain[-2] if direction == "head" else chain[1]

    if tip not in positions or prev not in positions:
        return None

    best_node = None
    best_angle = angle_limit_deg

    for nb in H.neighbors(tip):
        if nb in chain or nb not in positions:
            continue

        if direction == "head":
            angle = bend_angle_deg(positions[prev], positions[tip], positions[nb])
        else:
            angle = bend_angle_deg(positions[nb], positions[tip], positions[prev])

        if angle < best_angle:
            best_angle = angle
            best_node = int(nb)

    return best_node


def extract_force_chains(H, positions) -> List[List[int]]:
    chains: List[List[int]] = []
    visited_edges: set[EdgeKey] = set()

    # Start from the most central backbone edges.
    sorted_edges = sorted(
        H.edges(data=True),
        key=lambda item: item[2].get("ebc", 0.0),
        reverse=True,
    )

    for u, v, _ in sorted_edges:
        if edge_key(u, v) in visited_edges:
            continue

        chain = [int(u), int(v)]
        visited_edges.add(edge_key(u, v))

        # Grow in both directions from the seed edge.
        for direction in ("head", "tail"):
            while True:
                tip = chain[-1] if direction == "head" else chain[0]

                next_node = choose_next_node(
                    H,
                    positions,
                    chain,
                    direction,
                    angle_limit_deg,
                )

                if next_node is None:
                    break

                visited_edges.add(edge_key(tip, next_node))

                if direction == "head":
                    chain.append(next_node)
                else:
                    chain.insert(0, next_node)

        if len(chain) >= min_chain_length:
            chains.append(chain)

    return chains

In [8]:
# 3D Viz
##############
#LOG TRANSFORM
##############
def _get_log_transform(log_base: str, quantity_name: str):
    
    log_choice = str(log_base).lower().strip()

    if log_choice in ["log10", "base10", "10"]:
        return np.log10, "log10", f"log10({quantity_name})"
    if log_choice in ["ln", "natural", "e", "loge"]:
        return np.log, "ln", f"ln({quantity_name})"

    raise ValueError(f"log_base must be 'log10' or 'ln', not {log_base!r}")

def is_wrap_edge(a, b, pos, box_bounds):
    Lx = box_bounds["x"][1] - box_bounds["x"][0]
    Ly = box_bounds["y"][1] - box_bounds["y"][0]

    dx = abs(pos[a][0] - pos[b][0])
    dy = abs(pos[a][1] - pos[b][1])

    return dx > 0.5 * Lx or dy > 0.5 * Ly

def visualize_3d_chain_architecture(
    pos: PositionDict,
    chains: List[List[int]],
    G: nx.Graph,
    box_bounds,
    ebc: Dict[EdgeKey, float] | None = None,
    color_by: str = "force",
    save_html: str | Path | None = None,
    save_png: str | Path | None = None,
    background_contact_fraction: float = 1.0,
    random_seed: int = 1,
    gray_line_width: float = 1.0,
    gray_opacity: float = 0.35,
    chain_line_width: float = 5.0,
    z_range: tuple[float, float] | None = (0, 8),
    force_log_base: str = "log10",
    ebc_log_base: str = "log10",
    show_chain_nodes=False,
    node_size=2.5,
):
    # Plot full contact network plus selected force chains.

    # Gray lines: 
    #     Contacts that were not selected
    #     These are all non-selected contacts unless background_contact_fraction < 1.
    # Colored lines:
    #     Selected force-chain edges.

    # color_by:
    #     "force" colors selected chain edges by contact force.
    #     "ebc" colors selected chain edges by edge betweenness centrality.
    
    fig = go.Figure()

    
    #LOG TRANSFORM
    color_choice = str(color_by).lower().strip()
    if color_choice in ["force", "f"]:
        value_name = "edge force"
        log_func, log_label, colorbar_title = _get_log_transform(force_log_base, value_name)
        title = "Force chains colored by contact force"
    elif color_choice in ["ebc", "betweenness", "edge betweenness", "edge_betweenness"]:
        if ebc is None:
            raise ValueError("Pass the edge-betweenness dictionary as ebc=ebc when color_by='ebc'.")
        value_name = "EBC"
        log_func, log_label, colorbar_title = _get_log_transform(ebc_log_base, value_name)
        title = "Force chains colored by edge betweenness centrality"
    else:
        raise ValueError(f"color_by must be 'force' or 'ebc', not {color_by!r}")

    print(f"Coloring selected chain edges by: {value_name}")
    print(f"Color transform: {colorbar_title}")
    

    # no double grays!
    selected_edges = set()
    #for c in chains:
    #    for a, b in zip(c[:-1], c[1:]):

    
    # for c in chains:
    #     for a, b in zip(c[:-1], c[1:]):

    #         if is_wrap_edge(a, b, pos, box_bounds):
    #             continue

    #     # existing plotting code
            
    #         if is_wrap_edge(a, b, pos, box_bounds):
    #             print("Skipping wrap edge:", a, b)
    #             continue
                
    #         selected_edges.add(edge_key(a, b))



##########
    for c in chains:
        for a, b in zip(c[:-1], c[1:]):
    
            if is_wrap_edge(a, b, pos, box_bounds):
                print("Skipping wrap edge:", a, b)
                continue
    
            selected_edges.add(edge_key(a, b))
#########
    # all contacts except selected force-chains
    background_edges = [
        (a, b)
        for a, b in G.edges()
        if edge_key(a, b) not in selected_edges and a in pos and b in pos
    ]

    # sample gray network is too much
    if background_contact_fraction < 1.0 and len(background_edges) > 0:
        rng = np.random.default_rng(random_seed)
        n_keep = max(1, int(background_contact_fraction * len(background_edges)))
        keep_idx = rng.choice(len(background_edges), size=n_keep, replace=False)
        background_edges = [background_edges[i] for i in keep_idx]

    print(f"Gray background contacts shown: {len(background_edges):,}")

    # Gray
    bg_x, bg_y, bg_z = [], [], []
    for a, b in background_edges:
        if is_wrap_edge(a, b, pos, box_bounds):
            continue
        pa = pos[a]
        pb = pos[b]
        bg_x += [pa[0], pb[0], None]
        bg_y += [pa[1], pb[1], None]
        bg_z += [pa[2], pb[2], None]

    fig.add_trace(go.Scatter3d(
        x=bg_x,
        y=bg_y,
        z=bg_z,
        mode="lines",
        #edge_width = 1.0 + 8.0 * width_norm(log_value),
        #line=dict(color=color, width=edge_width),
        line=dict(color="rgb(135,135,135)", width=gray_line_width),
        
        opacity=gray_opacity,
        name="Background contacts",
        hoverinfo="skip",
        showlegend=False,
    ))
    

    # Selected chain edges
    edge_records = []
    # for chain_id, c in enumerate(chains):
    #     for a, b in zip(c[:-1], c[1:]):
    #         if not G.has_edge(a, b):
    #             continue
    #         if a not in pos or b not in pos:
    #             continue


###########
    for chain_id, c in enumerate(chains):
        for a, b in zip(c[:-1], c[1:]):

            if is_wrap_edge(a, b, pos, box_bounds):
                continue

            if not G.has_edge(a, b):
                continue
#########

            F = float(G[a][b].get("force", np.nan))
            B = np.nan if ebc is None else float(ebc.get(edge_key(a, b), np.nan))

            value = F if color_choice in ["force", "f"] else B
            if not np.isfinite(value) or value <= 0:
                continue
#####################
#LOG TRANSFORM VALUES
#####################
            log_value = log_func(value)
            edge_records.append((chain_id, a, b, F, B, value, log_value))

    if len(edge_records) == 0:
        print("No valid selected chain edges found.")
        return fig

    # Percentile clipping: prevents extreme edges from flattening the colormap.
    all_log_values = np.array([r[6] for r in edge_records])



    ##########
    #nodes
    ########
    width_norm = mcolors.Normalize(
    vmin=np.min(all_log_values),
    vmax=np.max(all_log_values)
    )
    #chain_nodes = set()
    
    chain_nodes = {}

    for chain_id, a, b, F, B, value, log_value in edge_records:
        if a not in chain_nodes:
            chain_nodes[a] = []
        if b not in chain_nodes:
            chain_nodes[b] = []

        chain_nodes[a].append(log_value)
        chain_nodes[b].append(log_value)
    
    #############
    
    width_norm = mcolors.Normalize(
    vmin=np.min(all_log_values),
    vmax=np.max(all_log_values)
)
    cmin = float(np.percentile(all_log_values, 5))
    cmax = float(np.percentile(all_log_values, 95))

    if np.isclose(cmin, cmax):
        cmin = float(np.min(all_log_values))
        cmax = float(np.max(all_log_values) + 1e-12)

    norm = mcolors.Normalize(vmin=cmin, vmax=cmax)
    cmap = cm.plasma

    # Draw selected force-chains
    for chain_id, a, b, F, B, value, log_value in edge_records:
        pa = pos[a]
        pb = pos[b]
        clipped_log_value = np.clip(log_value, cmin, cmax)
        color = mcolors.to_hex(cmap(norm(clipped_log_value)))

        if color_choice in ["force", "f"]:
            hover_text = (
                f"Chain {chain_id}"
                f"<br>F = {F:.3g}"
                f"<br>{log_label}(F) = {log_value:.3g}"
            )
        else:
            hover_text = (
                f"Chain {chain_id}"
                f"<br>EBC = {B:.3g}"
                f"<br>{log_label}(EBC) = {log_value:.3g}"
                f"<br>F = {F:.3g}"
            )
        #edge_width = 1.0 + 8.0 * width_norm(log_value)
        edge_width = 1.0 + 6.0 * (1.0 - width_norm(log_value))
        fig.add_trace(go.Scatter3d(
            x=[pa[0], pb[0]],
            y=[pa[1], pb[1]],
            z=[pa[2], pb[2]],
            mode="lines",
            line=dict(color=color, width=edge_width),
            #line=dict(color=color, width=chain_line_width),
            text=hover_text,
            hoverinfo="text",
            showlegend=False,
        ))

    
    #nodes
    if show_chain_nodes:

        node_x = []
        node_y = []
        node_z = []
        node_color = []
        
        for n, vals in chain_nodes.items():
        
            avg_val = np.mean(vals)
        
            clipped_val = np.clip(avg_val, cmin, cmax)
        
            node_x.append(pos[n][0])
            node_y.append(pos[n][1])
            node_z.append(pos[n][2])
        
            node_color.append(
                mcolors.to_hex(cmap(norm(clipped_val)))
            )
        #######################
    
        
        # Colored chain nodes
        node_x = []
        node_y = []
        node_z = []
        node_color = []
        
        for n, vals in chain_nodes.items():
            avg_val = np.mean(vals)
            clipped_val = np.clip(avg_val, cmin, cmax)
        
            node_x.append(pos[n][0])
            node_y.append(pos[n][1])
            node_z.append(pos[n][2])
            node_color.append(mcolors.to_hex(cmap(norm(clipped_val))))
        
        fig.add_trace(go.Scatter3d(
            x=node_x,
            y=node_y,
            z=node_z,
            mode="markers",
            marker=dict(
                size=2.5,
                color=node_color,
                opacity=1.0,
            ),
            hoverinfo="skip",
            showlegend=False,
        ))
    ###########################
    # colorbar
    fig.add_trace(go.Scatter3d(
        x=[None],
        y=[None],
        z=[None],
        mode="markers",
        marker=dict(
            size=0.01,
            color=[cmin, cmax],
            colorscale="Plasma_r",
            cmin=cmin,
            cmax=cmax,
            showscale=True,
            colorbar=dict(title=colorbar_title, thickness=20, len=0.8),
        ),
        hoverinfo="none",
        showlegend=False,
    ))

    # Grid is off otherwise grid looks like a mess
    zaxis_settings = dict(
        title="z",
        showgrid=False,
        zeroline=False,
        showbackground=False,
    )
    if z_range is not None:
        zaxis_settings["range"] = list(z_range)

    fig.update_layout(
        title=title,
        template="plotly_white",
        scene=dict(
            xaxis=dict(title="x", showgrid=False, zeroline=False, showbackground=False),
            yaxis=dict(title="y", showgrid=False, zeroline=False, showbackground=False),
            zaxis=zaxis_settings,
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, b=0, t=45),
        showlegend=False,
    )

    if save_html is not None:
        fig.write_html(str(save_html))
        print(f"Saved HTML: {save_html}")
    
    #this doesn't work yet.
    if save_png is not None:
        try:
            fig.write_image(str(save_png), width=2200, height=1800, scale=2)
            print(f"Saved PNG: {save_png}")
        except Exception as exc:
            print("PNG export failed. Install kaleido if you want PNG output.")
            print(f"Error: {exc}")

    fig.show()
    return fig


In [9]:
# 
# Run 


output_dir.mkdir(parents=True, exist_ok=True)

print("Reading particle positions...")
pos = read_lammps_positions(particle_file)
print(f"Loaded {len(pos):,} particle positions.")

print("Reading contacts...")
contacts = read_lammps_local_contacts(contact_file)
print(f"Loaded {len(contacts):,} positive-force contacts.")

box_bounds = manual_box_bounds
if box_bounds is None:
    box_bounds = read_lammps_box_bounds(particle_file)

# if remove_periodic_wrap_edges:
#     #removing wrap_edges -- may alter ebc calculation
#     #must reconsider this implementation
#     contacts = remove_periodic_boundary_edges(
#         contacts=contacts,
#         positions=pos,
#         box_bounds=box_bounds,
#         periodic_axes=periodic_axes,
#         wrap_fraction=periodic_wrap_fraction,
#     )



###
#fixed so wrap around contacts only affect visualization!
#must double check later

contacts_full = read_lammps_local_contacts(contact_file)

#build graph with full contacts
G = build_contact_graph(contacts_full)

#visualize without wrap-arounds

if remove_periodic_wrap_edges:
    contacts_plot = remove_periodic_boundary_edges(
        contacts=contacts_full,
        positions=pos,
        box_bounds=box_bounds,
        periodic_axes=periodic_axes,
        wrap_fraction=periodic_wrap_fraction,
    )
else:
    contacts_plot = contacts_full.copy()
###


print("Building contact graph...")
#G = build_contact_graph(contacts)
G = build_contact_graph(contacts_full)
print(f"Graph nodes: {G.number_of_nodes():,}")
print(f"Graph edges: {G.number_of_edges():,}")

print("Computing edge betweenness centrality...")
ebc = compute_edge_betweenness(G)

print("Selecting EBC backbone...")
H, ebc_threshold = make_backbone_graph(G, ebc, chain_percentile)
print(f"EBC threshold: {ebc_threshold:.4g}")
print(f"Backbone edges: {H.number_of_edges():,}")

print("Extracting force chains...")
chains = extract_force_chains(H, pos)
print(f"Extracted {len(chains):,} chains.")

force_html_path = output_dir / output_force_html
ebc_html_path = output_dir / output_ebc_html
png_path = None if output_png is None else output_dir / output_png

print("Making 3D figure 1: chains colored by force...")
fig_force = visualize_3d_chain_architecture(
    pos=pos,
    chains=chains,
    G=G,
    box_bounds=box_bounds,
    ebc=ebc,
    color_by="force",
    save_html=force_html_path,
    save_png=png_path,
    background_contact_fraction=background_contact_fraction,
    random_seed=random_seed,
    gray_line_width=gray_line_width,
    gray_opacity=gray_opacity,
    chain_line_width=chain_line_width,
    z_range=z_range,
    force_log_base=force_log_base,
    ebc_log_base=ebc_log_base,
    show_chain_nodes=True,
)

print("Making 3D figure 2: chains colored by EBC...")
fig_ebc = visualize_3d_chain_architecture(
    pos=pos,
    chains=chains,
    G=G,
    box_bounds=box_bounds,
    ebc=ebc,
    color_by="ebc",
    save_html=ebc_html_path,
    save_png=None,
    background_contact_fraction=background_contact_fraction,
    random_seed=random_seed,
    gray_line_width=gray_line_width,
    gray_opacity=gray_opacity,
    chain_line_width=chain_line_width,
    z_range=z_range,
    force_log_base=force_log_base,
    ebc_log_base=ebc_log_base,
    show_chain_nodes=True,
)

Reading particle positions...
Loaded 2,669 particle positions.
Reading contacts...
Loaded 3,423 positive-force contacts.
Periodic wrap-edge filtering:
  Periodic axes: ('x', 'y')
  Box bounds: {'x': (0.0, 20.0), 'y': (0.0, 20.0), 'z': (0.0, 60.0)}
  Wrap cutoff fraction: 0.5
  Before: 3,423 contacts
  Removed: 160 wraparound contacts
  After:  3,263 contacts
Building contact graph...
Graph nodes: 2,565
Graph edges: 3,423
Computing edge betweenness centrality...
Selecting EBC backbone...
EBC threshold: 0.01268
Backbone edges: 514
Extracting force chains...
Extracted 95 chains.
Making 3D figure 1: chains colored by force...
Coloring selected chain edges by: edge force
Color transform: ln(edge force)
Skipping wrap edge: 237 124
Gray background contacts shown: 3,177
Saved HTML: force_chain_output/force_chains_force_colored.html


Making 3D figure 2: chains colored by EBC...
Coloring selected chain edges by: EBC
Color transform: ln(EBC)
Skipping wrap edge: 237 124
Gray background contacts shown: 3,177
Saved HTML: force_chain_output/force_chains_ebc_colored.html


In [10]:
# 
# Plot without recomputing EBC
# 
# Change display settings only. This reuses positions, chains, Graph, and ebc from the pipeline above.

# Choose which version to plot: "force" or "ebc".
quick_color_by = "ebc"

background_contact_fraction = 0.5  
gray_line_width = 0.8
gray_opacity = 0.8
chain_line_width = 5.0

force_log_base = "log10"      # used when color_by = "force"
ebc_log_base = "log10"     # used when olor_by = "ebc"

fig_replot = visualize_3d_chain_architecture(
    pos=pos,
    chains=chains,
    G=G,
    box_bounds=box_bounds,
    ebc=ebc,
    color_by=quick_color_by,
    save_html=output_dir / f"force_chains_quick_replot_min{quick_color_by}.html",
    save_png=None,
    background_contact_fraction=background_contact_fraction,
    random_seed=random_seed,
    gray_line_width=gray_line_width,
    gray_opacity=gray_opacity,
    chain_line_width=chain_line_width,
    z_range=z_range,
    force_log_base=force_log_base,
    ebc_log_base=ebc_log_base,
    show_chain_nodes=False,
    node_size=3.0,
)

Coloring selected chain edges by: EBC
Color transform: log10(EBC)
Skipping wrap edge: 237 124
Gray background contacts shown: 1,588
Saved HTML: force_chain_output/force_chains_quick_replot_minebc.html
